# High-Level BART Tools with bartorch

This notebook demonstrates the high-level ops interface provided by **bartorch**: a PyTorch-native
Python wrapper for the Berkeley Advanced Reconstruction Toolbox (BART).

We cover:
- **Phantom generation** — simulated Shepp-Logan images and k-space data
- **FFT / IFFT** — forward and inverse non-unitary Fourier transform via BART
- **ESPIRiT calibration** — estimating coil sensitivity maps from multi-coil k-space
- **PICS reconstruction** — compressed sensing with L1-Wavelet regularisation
- **CFL file I/O** — reading and writing BART's native array format

> **Prerequisite:** BART must be installed as a system binary, e.g.
> `sudo apt-get install bart` on Ubuntu, or built from source following
> the [BART installation guide](https://mrirecon.github.io/bart/).
> When BART is not found, ops will raise `RuntimeError: BART executable not found`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import bartorch.ops as ops
from bartorch.utils.cfl import readcfl, writecfl

## 1. Phantom Generation

`ops.phantom()` wraps BART's `bart phantom` tool, which synthesises analytically
defined Shepp-Logan phantoms.  The result lives directly in a `BartTensor` — a
`torch.Tensor` subclass with `complex64` dtype and Fortran (column-major) strides
matching BART's internal memory layout.

In [ ]:
# Generate a 256×256 Shepp-Logan phantom
ph = ops.phantom([256, 256])
print(f"Phantom shape: {ph.shape}, dtype: {ph.dtype}")

Because `BartTensor` is a `torch.Tensor` subclass, every standard PyTorch operation
works on it out of the box: `.abs()`, `.real`, `.imag`, `.numpy()`, arithmetic, etc.
The tensor carries `complex64` values — one `complex float` per element — exactly
matching BART's native `complex float*` arrays.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(ph.squeeze().abs().numpy(), cmap='gray')
ax.set_title('Shepp-Logan Phantom')
ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Forward FFT (Simulated k-space)

`ops.fft()` calls `bart fft` with a bitmask that selects which dimensions to
transform.  `flags=3` means bits 0 and 1 are set, so the FFT is applied along
both spatial axes simultaneously — the standard 2-D k-space transform for MRI.

The log-magnitude of k-space is shown below; the characteristic bright centre
reflects the concentration of energy at low spatial frequencies.

In [ ]:
kspace = ops.fft(ph, flags=3)   # transform dims 0 and 1
print(f"K-space shape: {kspace.shape}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(np.log1p(kspace.squeeze().abs().numpy()), cmap='inferno')
ax.set_title('K-space (log magnitude)')
ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Inverse FFT (Zero-Filled Reconstruction)

`ops.ifft()` is a convenience wrapper around `bart fft -i`.  Applying it to
fully-sampled k-space is an exact round-trip: the reconstructed image should
be numerically identical to the original phantom (up to floating-point rounding).

In [ ]:
reco_zf = ops.ifft(kspace, flags=3)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(ph.squeeze().abs().numpy(), cmap='gray')
axes[0].set_title('Original phantom')
axes[0].axis('off')
axes[1].imshow(reco_zf.squeeze().abs().numpy(), cmap='gray')
axes[1].set_title('Zero-filled recon (|IFFT(FFT(x))|)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 4. Multi-Coil Phantom + ESPIRiT Calibration

Real MRI scanners use phased-array receiver coils.  BART can simulate multi-coil
data via `phantom -k -s <ncoils>`.  Each coil sees a spatially weighted version
of the object; the weights are the *sensitivity maps*.

**ESPIRiT** (`bart ecalib`) estimates those maps from the auto-calibration signal
(ACS) — a small, fully sampled region at the centre of k-space.  The calibration
region size is set by `calib_size`.

In [ ]:
# Generate an 8-coil k-space phantom
kspace_mc = ops.phantom([256, 256], kspace=True, ncoils=8)
print(f"Multi-coil k-space shape: {kspace_mc.shape}")

# Estimate sensitivity maps with ESPIRiT
sens = ops.ecalib(kspace_mc, calib_size=24)
print(f"Sensitivity maps shape: {sens.shape}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i in range(4):
    ax = axes[i]
    ax.imshow(sens[..., i, 0].abs().numpy(), cmap='viridis')
    ax.set_title(f'Coil {i+1} sensitivity')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. PICS Reconstruction

**PICS** (Parallel Imaging Compressed Sensing, `bart pics`) is BART's general
iterative reconstruction solver.  It minimises:

$$\hat{x} = \arg\min_x \frac{1}{2}\|A x - y\|_2^2 + \lambda\,\|W x\|_1$$

where $A$ is the SENSE encoding operator (sensitivity maps × FFT), $y$ is the
measured k-space, $W$ is a sparsifying transform (here: wavelets), and
$\lambda$ controls regularisation strength.

The comparison below shows the zero-filled root-sum-of-squares (RSS) reference
alongside the PICS result.  For fully-sampled data the difference is subtle;
the benefit is large for undersampled acquisitions.

In [ ]:
# Reconstruct with L1-Wavelet regularisation
reco = ops.pics(kspace_mc, sens, lambda_=0.005, iter_=30, wav=True)
print(f"Reconstruction shape: {reco.shape}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
# reference: zero-filled coil combination (RSS)
rss = (kspace_mc.abs()**2).sum(dim=-1).sqrt()
rss_img = ops.ifft(rss, flags=3)
axes[0].imshow(rss_img.squeeze().abs().numpy(), cmap='gray')
axes[0].set_title('Zero-filled RSS')
axes[0].axis('off')
axes[1].imshow(reco.squeeze().abs().numpy(), cmap='gray')
axes[1].set_title('PICS L1-Wavelet')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 6. Saving and Loading CFL Files

BART's native array format is **CFL**: a pair of files `<name>.hdr` (ASCII header
with dimensions) and `<name>.cfl` (raw `complex float` binary data, Fortran order).
`bartorch.utils.cfl` provides `writecfl` / `readcfl` that are fully compatible
with BART's own file I/O.  This lets you exchange data with command-line BART
scripts, MATLAB, or other tools.

In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, 'phantom_test')
    writecfl(path, ph.numpy())
    loaded = readcfl(path)
    print(f"Written shape: {ph.shape}, Loaded shape: {loaded.shape}")
    print(f"Max abs error: {np.abs(ph.numpy() - loaded).max():.2e}")

## Summary

This notebook walked through the complete bartorch high-level ops interface:

| Step | Function | BART tool |
|------|----------|-----------|
| Phantom generation | `ops.phantom()` | `bart phantom` |
| Forward FFT | `ops.fft()` | `bart fft` |
| Inverse FFT | `ops.ifft()` | `bart fft -i` |
| Coil calibration | `ops.ecalib()` | `bart ecalib` |
| CS reconstruction | `ops.pics()` | `bart pics` |
| CFL file I/O | `readcfl` / `writecfl` | — |

All ops currently use BART as a subprocess with CFL temp files in `/dev/shm`
(RAM-backed tmpfs on Linux).  Once the C++ extension is built, they will
automatically switch to the zero-copy hot path.  See
`02_library_internals.ipynb` for a deep dive into the dispatch mechanism.